# 02a — Canonical month-end archive replay

Materialise one canonical Silver state per archive month, then run the DQ
and Gold notebooks for that snapshot. Archive ZIP exports are treated as
complete table snapshots, so each table uses its latest available export
on or before the month's final available export date.

This avoids loading every daily export. Primary-key duplicates inside the
selected export are resolved with `row_number()`. Row-level `export_date`
is retained in `silver.slv_<table>` and checked before Gold is invoked.


In [1]:
ARCHIVE_SCHEMA = "archived"
SILVER_SCHEMA = "silver"
CFG_NOTEBOOK_NAME = "00_setup_cfg 02 03"
SCHEMA_CSV_PATH = "Files/cfg_files/schema_definition.csv"
AUDIT_TABLE = "monitoring.cfg_silver_export_load"
ARCHIVE_PREFIXES = ("archived_",)
EXCLUDED_ARCHIVE_TABLES = {"archived_audit"}  # Raw change log has no schema contract.

# Optional YYYY-MM-DD. Any date selects that calendar month's canonical
# final export. Blank processes all archive months from the minimum to the
# maximum available export date.
BATCH_EXPORT_DATE = ""

RUN_GOLD_AT_MONTH_END = True
DQ_NOTEBOOK_NAME = "03_silver_business_rules 02 03"
GOLD_NOTEBOOK_NAME = "04_gold_model 02 03"
NOTEBOOK_TIMEOUT_SECONDS = 1800
STRICT_SCHEMA = True
FAIL_ON_TABLE_ERROR = True

# Diagnostics print only identifiers, dates and counts—not child details.
VERBOSE_DIAGNOSTICS = True
DIAGNOSTIC_KEY_SAMPLE_SIZE = 5

DATE_FORMATS = ["yyyy-MM-dd", "dd/MM/yyyy", "yyyy-MM-dd'T'HH:mm:ss"]
TIME_PARSER_POLICY = "CORRECTED"
TIMESTAMP_FORMATS = [
    "yyyy-MM-dd",
    "yyyy-MM-dd HH:mm:ss.SSSSSS",
    "yyyy-MM-dd HH:mm:ss.SSS",
    "yyyy-MM-dd HH:mm:ss.S",
    "yyyy-MM-dd HH:mm:ss",
    "yyyy-MM-dd'T'HH:mm:ss.SSSSSS",
    "yyyy-MM-dd'T'HH:mm:ss.SSS",
    "yyyy-MM-dd'T'HH:mm:ss.S",
    "yyyy-MM-dd'T'HH:mm:ss",
    "yyyy-MM-dd'T'HH:mm:ss.SSSXXX",
    "yyyy-MM-dd'T'HH:mm:ss.SSSSSSXXX",
]


StatementMeta(, 4c7ffa86-84fd-48a4-afa1-31c8122b53c3, 3, Finished, Available, Finished, False)

In [2]:
# %run '/Notebooks/99_common_library 02 03.ipynb'

StatementMeta(, 4c7ffa86-84fd-48a4-afa1-31c8122b53c3, 4, Finished, Available, Finished, False)

In [3]:
import re
from bisect import bisect_right
import uuid
from collections import defaultdict
from datetime import datetime
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.types import (
    BooleanType, IntegerType, LongType, StringType, StructField, StructType, TimestampType
)
from pyspark.sql.window import Window

RUN_ID = str(uuid.uuid4())
STARTED_AT = datetime.utcnow()
spark.conf.set("spark.sql.legacy.timeParserPolicy", TIME_PARSER_POLICY)


def qident(value):
    return "`" + str(value).replace("`", "``") + "`"


def normalise(value):
    return re.sub(r"[^a-z0-9]", "", (value or "").lower())


def append_rows(table_name, rows, schema):
    if rows:
        spark.createDataFrame(rows, schema).write.format("delta").mode("append").saveAsTable(table_name)


def map_data_type(pg_type):
    value = (pg_type or "").lower().strip()
    if "[]" in value:
        return "ARRAY<STRING>"
    if any(token in value for token in ("uuid", "json", "text", "character", "varchar")):
        return "STRING"
    if value in {"smallint", "int2", "integer", "int", "int4"}:
        return "INT"
    if value in {"bigint", "int8"}:
        return "BIGINT"
    match = re.search(r"(?:numeric|decimal)\s*\((\d+)\s*,\s*(\d+)\)", value)
    if match:
        precision = min(int(match.group(1)), 38)
        scale = min(int(match.group(2)), precision)
        return f"DECIMAL({precision},{scale})"
    if "numeric" in value or "decimal" in value:
        return "DECIMAL(38,18)"
    if any(token in value for token in ("double", "float", "real")):
        return "DOUBLE"
    if "boolean" in value or value == "bool":
        return "BOOLEAN"
    if value == "date":
        return "DATE"
    if "timestamp" in value:
        return "TIMESTAMP"
    return "STRING"


def first_parsed(column, formats, parser):
    return F.coalesce(*[parser(column, fmt) for fmt in formats])


def cast_column(frame, definition):
    name = definition["column_name"]
    spark_type = map_data_type(definition["data_type"])
    if name not in frame.columns:
        return F.lit(None).cast(spark_type).alias(name)
    source = F.col(qident(name))
    if spark_type == "BOOLEAN":
        clean = F.lower(F.trim(source.cast("string")))
        return (F.when(clean.isin("true", "t", "1", "yes", "y"), F.lit(True))
            .when(clean.isin("false", "f", "0", "no", "n"), F.lit(False))
            .otherwise(F.lit(None).cast("boolean")).alias(name))
    if spark_type == "DATE":
        return first_parsed(source.cast("string"), DATE_FORMATS, F.to_date).alias(name)
    if spark_type == "TIMESTAMP":
        return first_parsed(source.cast("string"), TIMESTAMP_FORMATS, F.to_timestamp).alias(name)
    if spark_type.startswith("DECIMAL") or spark_type in {"INT", "BIGINT", "DOUBLE"}:
        return F.regexp_replace(source.cast("string"), r"[^0-9eE+\.\-]", "").cast(spark_type).alias(name)
    if spark_type == "ARRAY<STRING>":
        return F.when(source.isNull(), F.lit(None).cast("array<string>"))             .otherwise(F.split(F.regexp_replace(source.cast("string"), r"^[\{\[]|[\}\]]$", ""), r"\s*,\s*")).alias(name)
    return F.trim(source.cast("string")).alias(name)


def resolve_contract(physical_table, prefixes):
    base = physical_table.lower()
    for prefix in prefixes:
        if base.startswith(prefix):
            base = base[len(prefix):]
    matches = contracts_by_table.get(normalise(base), [])
    if len(matches) == 1:
        return matches[0]
    if len(matches) > 1:
        raise ValueError(f"Ambiguous table contract for {physical_table}: {matches}")
    return None


def primary_key_columns(schema_cols):
    """Return the ordered business key defined by the schema contract."""
    return [
        column["column_name"] for column in schema_cols
        if (column.get("is_primary_key") or "").upper() == "YES"
    ]


def deduplicate_frame(frame, schema_cols):
    """Keep one row per contracted PK without triggering count jobs here."""
    key_columns = primary_key_columns(schema_cols)
    if not key_columns:
        return frame, key_columns
    if "_archive_load_ts" in frame.columns:
        ordering = F.col("_archive_load_ts").cast("timestamp").desc_nulls_last()
    elif "_ingestion_timestamp" in frame.columns:
        ordering = F.col("_ingestion_timestamp").cast("timestamp").desc_nulls_last()
    else:
        ordering = F.col(qident("export_date")).cast("timestamp").desc_nulls_last()
    window = Window.partitionBy(
        *[F.col(qident(column)) for column in key_columns]
    ).orderBy(ordering)
    ranked = frame.withColumn("_silver_row_number", F.row_number().over(window))
    return ranked.where(F.col("_silver_row_number") == 1).drop("_silver_row_number"), key_columns


def diagnostic_key_sample(frame, key_columns):
    """Collect a small identifier-only sample for operational tracing."""
    if not VERBOSE_DIAGNOSTICS or not key_columns:
        return []
    return [
        row.asDict(recursive=True)
        for row in frame.select(*[F.col(qident(column)) for column in key_columns])
            .limit(DIAGNOSTIC_KEY_SAMPLE_SIZE).collect()
    ]


def print_load_diagnostics(source_table, target_table, snapshot_date, source_date,
                           raw_frame, formatted_frame, key_columns,
                           source_count, written_count, duplicate_count,
                           formatted_export_non_null):
    """Print dates, counts and PK identifiers needed to diagnose a batch."""
    if not VERBOSE_DIAGNOSTICS:
        return
    raw_export_type = next(
        field.dataType.simpleString() for field in raw_frame.schema.fields
        if field.name == "export_date"
    )
    formatted_dates = [
        str(row["export_date"])
        for row in formatted_frame.select(F.to_date("export_date").alias("export_date"))
            .where(F.col("export_date").isNotNull()).distinct().orderBy("export_date").collect()
    ]
    print(f"  Source: {source_table}; selected export={source_date}; raw type={raw_export_type}")
    print(f"  Target: {target_table}; Gold snapshot={snapshot_date}")
    print(f"  Rows: source={source_count:,}; written={written_count:,}; duplicates removed={duplicate_count:,}")
    print(f"  Primary keys: {key_columns or ['<none>']}")
    print(f"  Primary-key sample: {diagnostic_key_sample(formatted_frame, key_columns)}")
    print(
        f"  Silver export_date: non-null={formatted_export_non_null:,}; "
        f"null={written_count - formatted_export_non_null:,}; values={formatted_dates}"
    )


def format_frame(frame, schema_cols, source_kind, source_table):
    expressions = [cast_column(frame, definition) for definition in schema_cols]
    return (frame.select(*expressions)
        .withColumn("_record_source", F.lit(source_kind))
        .withColumn("_source_table", F.lit(source_table))
        .withColumn("_silver_run_id", F.lit(RUN_ID))
        .withColumn("_silver_load_ts", F.current_timestamp()))


AUDIT_SCHEMA = StructType([
    StructField("source_kind", StringType(), False),
    StructField("source_schema", StringType(), False),
    StructField("source_table", StringType(), False),
    StructField("target_table", StringType(), True),
    StructField("export_date", TimestampType(), False),
    StructField("status", StringType(), False),
    StructField("reload", BooleanType(), False),
    StructField("attempt_count", IntegerType(), False),
    StructField("run_id", StringType(), True),
    StructField("rows_read", LongType(), True),
    StructField("rows_written", LongType(), True),
    StructField("duplicate_key_count", LongType(), True),
    StructField("started_at", TimestampType(), True),
    StructField("ended_at", TimestampType(), True),
    StructField("error_message", StringType(), True),
    StructField("last_updated_at", TimestampType(), True),
])


def audit_record(source_kind, source_schema, source_table, export_date):
    rows = (spark.table(AUDIT_TABLE)
        .where((F.col("source_kind") == source_kind)
            & (F.col("source_schema") == source_schema)
            & (F.col("source_table") == source_table)
            & (F.col("export_date") == F.lit(export_date).cast("timestamp")))
        .limit(1).collect())
    return rows[0].asDict() if rows else None


def should_skip(source_kind, source_schema, source_table, export_date):
    record = audit_record(source_kind, source_schema, source_table, export_date)
    return bool(record and record["status"] == "SUCCESS" and not record["reload"])


def audit_begin(source_kind, source_schema, source_table, target_table, export_date):
    now = datetime.utcnow()
    existing = audit_record(source_kind, source_schema, source_table, export_date)
    attempt_count = int(existing["attempt_count"] or 0) + 1 if existing else 1
    row = [(source_kind, source_schema, source_table, target_table, export_date, "RUNNING",
            bool(existing["reload"]) if existing else False, attempt_count, RUN_ID,
            None, None, None, now, None, None, now)]
    source = spark.createDataFrame(row, AUDIT_SCHEMA)
    target = DeltaTable.forName(spark, AUDIT_TABLE)
    condition = " AND ".join([
        "t.source_kind = s.source_kind", "t.source_schema = s.source_schema",
        "t.source_table = s.source_table", "t.export_date = s.export_date",
    ])
    (target.alias("t").merge(source.alias("s"), condition)
        .whenMatchedUpdate(set={
            "target_table": "s.target_table", "status": "s.status",
            "attempt_count": "s.attempt_count", "run_id": "s.run_id",
            "started_at": "s.started_at", "ended_at": "s.ended_at",
            "error_message": "s.error_message", "last_updated_at": "s.last_updated_at",
        }).whenNotMatchedInsertAll().execute())


def audit_finish(source_kind, source_schema, source_table, target_table, export_date,
                 status, rows_read=0, rows_written=0, duplicate_count=0, error_message=None):
    now = datetime.utcnow()
    existing = audit_record(source_kind, source_schema, source_table, export_date) or {}
    row = [(source_kind, source_schema, source_table, target_table, export_date, status,
            False if status == "SUCCESS" else bool(existing.get("reload", False)),
            int(existing.get("attempt_count") or 1), RUN_ID, int(rows_read), int(rows_written),
            int(duplicate_count), existing.get("started_at") or now, now,
            error_message[:4000] if error_message else None, now)]
    source = spark.createDataFrame(row, AUDIT_SCHEMA)
    target = DeltaTable.forName(spark, AUDIT_TABLE)
    condition = " AND ".join([
        "t.source_kind = s.source_kind", "t.source_schema = s.source_schema",
        "t.source_table = s.source_table", "t.export_date = s.export_date",
    ])
    (target.alias("t").merge(source.alias("s"), condition)
        .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())


StatementMeta(, 4c7ffa86-84fd-48a4-afa1-31c8122b53c3, 5, Finished, Available, Finished, False)

In [4]:
# spark.sql(f"CREATE SCHEMA IF NOT EXISTS {qident(SILVER_SCHEMA)}")


StatementMeta(, 4c7ffa86-84fd-48a4-afa1-31c8122b53c3, 6, Finished, Available, Finished, False)

In [7]:
# %%sql
# SET legacy_time_parser_policy = legacy;


StatementMeta(, 4c7ffa86-84fd-48a4-afa1-31c8122b53c3, 9, Finished, Available, Finished, False)

In [8]:

# cfg_result = mssparkutils.notebook.run(
#                 CFG_NOTEBOOK_NAME,
#                 NOTEBOOK_TIMEOUT_SECONDS,
#                 {"AUDIT_TABLE": "{AUDIT_TABLE}"}
#             )


StatementMeta(, 4c7ffa86-84fd-48a4-afa1-31c8122b53c3, 10, Finished, Available, Finished, False)

In [ ]:

    
append_rows(
    "monitoring.cfg_pipeline_run",
    [(RUN_ID, "02a_archive_silver", "SILVER", "ARCHIVE", STARTED_AT, None, "RUNNING", 0, 0, 0, 0, None)],
    "run_id string,pipeline_name string,layer string,source_kind string,started_at timestamp,ended_at timestamp,status string,tables_succeeded int,tables_failed int,rows_read long,rows_written long,error_message string",
)


StatementMeta(, 4c7ffa86-84fd-48a4-afa1-31c8122b53c3, -1, Cancelled, , Cancelled, True)

In [ ]:
required_schema_columns = {
    "schema_name", "table_name", "ordinal_position", "column_name", "data_type",
    "is_nullable", "is_primary_key", "referenced_schema", "referenced_table", "referenced_column",
}
schema_df = (spark.read.format("csv").option("header", "true").option("quote", '"')
    .option("escape", '"').load(SCHEMA_CSV_PATH))
missing_metadata_columns = required_schema_columns - set(schema_df.columns)
if missing_metadata_columns:
    raise ValueError(f"schema_definition.csv is missing: {sorted(missing_metadata_columns)}")

schema_rows = [row.asDict(recursive=True) for row in schema_df.collect()]
schema_df.withColumn("contract_loaded_at", F.current_timestamp()).write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("monitoring.cfg_schema_contract_column")

contracts = defaultdict(list)
for row in schema_rows:
    if row.get("schema_name") and row.get("table_name") and row.get("column_name"):
        row["ordinal_position"] = int(row.get("ordinal_position") or 999999)
        contracts[(row["schema_name"].lower(), row["table_name"].lower())].append(row)
for key in contracts:
    contracts[key].sort(key=lambda item: item["ordinal_position"])

contracts_by_table = defaultdict(list)
for key in contracts:
    contracts_by_table[normalise(key[1])].append(key)

print(f"Loaded {len(schema_rows):,} column definitions for {len(contracts):,} tables")


StatementMeta(, 4c7ffa86-84fd-48a4-afa1-31c8122b53c3, -1, Cancelled, , Cancelled, True)

In [ ]:
# Resolve every archive table to its contract once. We also collect each
# table's available export dates once, avoiding a max-date Spark job for
# every table in every month.
table_rows = spark.sql(f"SHOW TABLES IN {qident(ARCHIVE_SCHEMA)}").collect()
physical_tables = sorted(
    row.tableName for row in table_rows
    if not row.isTemporary
    and not row.tableName.lower().startswith("cfg_")
    and row.tableName.lower() not in {
        name.lower() for name in EXCLUDED_ARCHIVE_TABLES
    }
)

excluded_present = sorted(
    row.tableName for row in table_rows
    if row.tableName.lower() in {
        name.lower() for name in EXCLUDED_ARCHIVE_TABLES
    }
)
if excluded_present:
    print(f"Excluded raw archive-only tables: {excluded_present}")

source_tables = []
all_available_dates = set()
for physical_table in physical_tables:
    source_table = f"{ARCHIVE_SCHEMA}.{physical_table}"
    source_frame = spark.table(source_table)
    if "export_date" not in source_frame.columns:
        raise ValueError(f"{source_table} has no row-level export_date")

    contract_key = resolve_contract(physical_table, ARCHIVE_PREFIXES)
    if contract_key is None:
        message = f"No schema contract for {source_table}"
        if STRICT_SCHEMA:
            raise ValueError(message)
        print(f"WARN {message}")
        continue

    _, contract_table = contract_key
    target_table = f"{SILVER_SCHEMA}.slv_{contract_table}"
    table_dates = sorted(
        row["export_date"]
        for row in source_frame
            .select(F.to_date("export_date").alias("export_date"))
            .where(F.col("export_date").isNotNull())
            .distinct().collect()
    )
    if not table_dates:
        print(f"WARN {source_table} has no valid export_date values")
        continue

    all_available_dates.update(table_dates)
    source_tables.append({
        "physical_table": physical_table,
        "source_table": source_table,
        "target_table": target_table,
        "schema_cols": contracts[contract_key],
        "available_dates": table_dates,
    })
    if VERBOSE_DIAGNOSTICS:
        print(
            f"TABLE {source_table} -> {target_table}: "
            f"exports={len(table_dates):,}, min={table_dates[0]}, max={table_dates[-1]}"
        )

if not source_tables or not all_available_dates:
    raise ValueError(f"No replayable archive tables found in {ARCHIVE_SCHEMA}")

available_dates = sorted(all_available_dates)
minimum_export_date = available_dates[0]
maximum_export_date = available_dates[-1]

# The final available export in each calendar month is the canonical
# snapshot date. A table that was not exported on that exact date carries
# forward its own latest available full snapshot on or before the date.
month_last_dates = {}
for export_date in available_dates:
    month_last_dates[(export_date.year, export_date.month)] = export_date
month_end_dates = sorted(month_last_dates.values())

if BATCH_EXPORT_DATE:
    requested_date = datetime.strptime(BATCH_EXPORT_DATE, "%Y-%m-%d").date()
    requested_month = (requested_date.year, requested_date.month)
    if requested_month not in month_last_dates:
        raise ValueError(f"No archive exports found in requested month {requested_date:%Y-%m}")
    canonical_date = month_last_dates[requested_month]
    month_end_dates = [canonical_date]
    print(
        f"Requested {requested_date}; canonical month-end export is {canonical_date}"
    )

print(
    f"Archive export range: {minimum_export_date} to {maximum_export_date}; "
    f"{len(available_dates):,} distinct export days"
)
print(
    f"Month-end batches: {len(month_end_dates):,}; "
    f"{', '.join(str(value) for value in month_end_dates)}"
)


StatementMeta(, 4c7ffa86-84fd-48a4-afa1-31c8122b53c3, -1, Cancelled, , Cancelled, True)

In [ ]:
from notebookutils import mssparkutils

SOURCE_KIND = "ARCHIVE_MONTH_END"
METRIC_SCHEMA = "run_id string,layer string,source_kind string,source_object string,target_object string,rows_read long,rows_written long,duplicate_key_count long,null_primary_key_count long,recorded_at timestamp"
ok = failed = skipped = total_read = total_written = 0
errors = []


def month_end_record(snapshot_date):
    """Return the orchestration state for one canonical month-end."""
    rows = (spark.table("monitoring.cfg_month_end_gold_run")
        .where(F.col("snapshot_date") == F.lit(snapshot_date).cast("date"))
        .limit(1).collect())
    return rows[0].asDict() if rows else None


def update_month_end(snapshot_date, status, dq_result=None,
                     gold_result=None, error_message=None):
    """Upsert the DQ/Gold orchestration state for a snapshot."""
    now = datetime.utcnow()
    existing = month_end_record(snapshot_date) or {}
    attempt_count = int(existing.get("attempt_count") or 0) + (
        1 if status == "RUNNING" else 0
    )
    row = spark.createDataFrame([(
        snapshot_date,
        status,
        False if status in ("RUNNING", "SUCCESS") else bool(existing.get("reload", False)),
        attempt_count,
        RUN_ID,
        now if status == "RUNNING" else existing.get("started_at"),
        None if status == "RUNNING" else now,
        dq_result,
        gold_result,
        error_message[:4000] if error_message else None,
        now,
    )], "snapshot_date date,status string,reload boolean,attempt_count int,run_id string,started_at timestamp,ended_at timestamp,dq_result string,gold_result string,error_message string,last_updated_at timestamp")
    target = DeltaTable.forName(spark, "monitoring.cfg_month_end_gold_run")
    (target.alias("t").merge(row.alias("s"), "t.snapshot_date = s.snapshot_date")
        .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())


def latest_table_export(table_dates, snapshot_date):
    """Choose a table's latest full export on or before the snapshot."""
    position = bisect_right(table_dates, snapshot_date) - 1
    return table_dates[position] if position >= 0 else None


def materialise_table_month_end(table_info, snapshot_date):
    """Overwrite one Silver table with its canonical month-end state."""
    source_table = table_info["source_table"]
    target_table = table_info["target_table"]
    schema_cols = table_info["schema_cols"]
    source_date = latest_table_export(
        table_info["available_dates"], snapshot_date
    )
    if source_date is None:
        print(f"SKIP {source_table}: no export on or before {snapshot_date}")
        return 0, 0, 0

    audit_timestamp = datetime.combine(snapshot_date, datetime.min.time())
    audit_begin(
        SOURCE_KIND, ARCHIVE_SCHEMA, source_table,
        target_table, audit_timestamp,
    )
    formatted = None
    try:
        # Each dated archive file is a complete snapshot. Selecting only
        # the final available file avoids scanning every daily snapshot.
        raw_frame = (spark.table(source_table)
            .where(F.to_date("export_date") == F.lit(source_date)))
        source_count = raw_frame.count()
        if source_count == 0:
            raise ValueError(f"No rows found for selected export {source_date}")

        deduplicated, key_columns = deduplicate_frame(raw_frame, schema_cols)

        # Persist so validation, diagnostics and the Delta write reuse the
        # same conformed result instead of recomputing the Spark plan.
        formatted = format_frame(
            deduplicated, schema_cols, SOURCE_KIND, source_table
        ).persist()
        written = formatted.count()
        duplicate_count = source_count - written
        formatted_export_non_null = formatted.where(
            F.col("export_date").isNotNull()
        ).count()
        if formatted_export_non_null != written:
            raise ValueError(
                f"export_date parse failure in {source_table}: "
                f"{written - formatted_export_non_null} of {written} rows are null"
            )

        print(f"MONTH END {snapshot_date}: {source_table} -> {target_table}")
        print_load_diagnostics(
            source_table, target_table, snapshot_date, source_date,
            raw_frame, formatted, key_columns, source_count, written,
            duplicate_count, formatted_export_non_null,
        )

        (formatted.write.format("delta").mode("overwrite")
            .option("overwriteSchema", "true").saveAsTable(target_table))

        if VERBOSE_DIAGNOSTICS:
            target_stats = spark.table(target_table).agg(
                F.count(F.lit(1)).alias("rows"),
                F.sum(F.when(F.col("export_date").isNull(), 1).otherwise(0)).alias("null_dates"),
                F.min("export_date").alias("min_export_date"),
                F.max("export_date").alias("max_export_date"),
            ).first().asDict()
            print(f"  TARGET CHECK {target_table}: {target_stats}")

        audit_finish(
            SOURCE_KIND, ARCHIVE_SCHEMA, source_table, target_table,
            audit_timestamp, "SUCCESS", source_count, written,
            duplicate_count,
        )
        append_rows(
            "monitoring.cfg_table_load_metric",
            [(RUN_ID, "SILVER", SOURCE_KIND, source_table, target_table,
              source_count, written, duplicate_count, None, datetime.utcnow())],
            METRIC_SCHEMA,
        )
        return source_count, written, duplicate_count
    except Exception as exc:
        audit_finish(
            SOURCE_KIND, ARCHIVE_SCHEMA, source_table, target_table,
            audit_timestamp, "FAILED", error_message=str(exc)[:4000],
        )
        raise
    finally:
        if formatted is not None:
            formatted.unpersist()


for snapshot_date in month_end_dates:
    existing = month_end_record(snapshot_date)
    if existing and existing["status"] == "SUCCESS" and not existing["reload"]:
        skipped += 1
        print(f"SKIP MONTH {snapshot_date}: Silver, DQ and Gold already successful")
        continue

    print(f"\n=== PROCESSING CANONICAL MONTH END {snapshot_date} ===")
    update_month_end(snapshot_date, "RUNNING")
    month_errors = []

    # Rebuild every Silver table for a month being retried. Individual
    # table audit rows cannot be used to skip here because shared Silver
    # may currently contain a later month's state.
    for table_info in source_tables:
        try:
            rows_read, rows_written, _ = materialise_table_month_end(
                table_info, snapshot_date
            )
            ok += 1
            total_read += rows_read
            total_written += rows_written
        except Exception as exc:
            message = str(exc)[:4000]
            failed += 1
            month_errors.append(f"{table_info['source_table']}: {message}")
            print(f"FAILED {table_info['source_table']} @ {snapshot_date}: {message}")

    if month_errors:
        message = " | ".join(month_errors)[:4000]
        errors.append(f"{snapshot_date}: {message}")
        update_month_end(snapshot_date, "FAILED", error_message=message)
        if FAIL_ON_TABLE_ERROR:
            break
        continue

    if RUN_GOLD_AT_MONTH_END:
        try:
            dq_result = mssparkutils.notebook.run(
                DQ_NOTEBOOK_NAME, NOTEBOOK_TIMEOUT_SECONDS
            )
            gold_result = mssparkutils.notebook.run(
                GOLD_NOTEBOOK_NAME,
                NOTEBOOK_TIMEOUT_SECONDS,
                {"AS_OF_DATE": snapshot_date.isoformat()},
            )
            update_month_end(
                snapshot_date, "SUCCESS", str(dq_result), str(gold_result)
            )
            print(f"MONTH END COMPLETE {snapshot_date}: DQ and Gold succeeded")
        except Exception as exc:
            message = str(exc)[:4000]
            errors.append(f"{snapshot_date} DQ/Gold: {message}")
            update_month_end(snapshot_date, "FAILED", error_message=message)
            print(f"MONTH END FAILED {snapshot_date}: {message}")
            if FAIL_ON_TABLE_ERROR:
                break
    else:
        update_month_end(snapshot_date, "SUCCESS")

status = "FAILED" if errors else "SUCCESS"
error_text = " | ".join(errors)[:4000] if errors else None
error_sql = "NULL" if error_text is None else "'" + error_text.replace("'", "''") + "'"
spark.sql(f"""UPDATE monitoring.cfg_pipeline_run
SET ended_at=current_timestamp(), status='{status}',
    tables_succeeded={ok}, tables_failed={failed},
    rows_read={total_read}, rows_written={total_written},
    error_message={error_sql}
WHERE run_id='{RUN_ID}'""")
print(
    f"Archive month-end run {RUN_ID}: {status}; "
    f"loaded={ok}, skipped_months={skipped}, failed={failed}"
)
if errors and FAIL_ON_TABLE_ERROR:
    raise RuntimeError(error_text)


StatementMeta(, 4c7ffa86-84fd-48a4-afa1-31c8122b53c3, -1, Cancelled, , Cancelled, True)